# Tests de mini-funcionalidades de infer_trips_from_traces

## Bloque 0. Preparación

### Imports generales

In [1]:
import json
from copy import deepcopy

import numpy as np
import pandas as pd

### Imports del módulo

In [2]:
from pylondrina.datasets import TraceDataset, TripDataset
from pylondrina.errors import InferenceError, SchemaError
from pylondrina.reports import Issue, InferenceReport
from pylondrina.schema import DomainSpec, FieldSpec, TraceSchema, TripSchema

from pylondrina.transforms.inference import (
    InferTripsOptions,
    _resolve_infer_request,
    _build_point_candidates,
    _build_sequential_clusters,
    _build_cluster_candidates,
    _evaluate_candidates,
    _materialize_trip_dataframe,
    _enrich_trip_dataframe,
    _build_inference_outputs,
    _normalize_optional_number,
    _normalize_propagate_trace_fields,
    _prepare_trace_workframe,
    _cluster_record,
    _same_place_mask,
    _haversine_meters,
    _safe_distance_meters,
    _safe_time_gap_seconds,
    _empty_candidates_frame,
    _expected_propagated_columns,
    _derive_h3_series,
    _normalize_output_categorical_field,
    _build_issues_summary,
    _summarize_prior_events,
    _time_range_summary,
    _to_json_safe,
)

### Helpers de apoyo para test

In [3]:
def show_ok(label: str):
    print(f"OK - {label}")


def issue_codes(issues):
    return [issue.code for issue in issues]


def assert_has_code(issues, code: str):
    codes = issue_codes(issues)
    assert code in codes, f"No se encontró {code}. Codes emitidos: {codes}"


def make_issue(level: str, code: str, message: str = "dummy") -> Issue:
    return Issue(level=level, code=code, message=message)


def make_trace_schema_min() -> TraceSchema:
    fields = {
        "point_id": FieldSpec(name="point_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "time_utc": FieldSpec(name="time_utc", dtype="datetime", required=True),
        "latitude": FieldSpec(
            name="latitude",
            dtype="float",
            required=True,
            constraints={"range": {"min": -90, "max": 90}},
        ),
        "longitude": FieldSpec(
            name="longitude",
            dtype="float",
            required=True,
            constraints={"range": {"min": -180, "max": 180}},
        ),
        "location_ref": FieldSpec(name="location_ref", dtype="string", required=False),
        "poi_cat": FieldSpec(name="poi_cat", dtype="string", required=False),
    }
    return TraceSchema(
        version="trace-v1",
        fields=fields,
        required=["point_id", "user_id", "time_utc", "latitude", "longitude"],
    )


def make_trip_schema_min(include_propagated_categoricals: bool = False) -> TripSchema:
    base = {
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=True),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=True),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=True),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
    }

    if include_propagated_categoricals:
        base["origin_poi_cat"] = FieldSpec(
            name="origin_poi_cat",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        )
        base["destination_poi_cat"] = FieldSpec(
            name="destination_poi_cat",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True),
        )

    return TripSchema(
        version="trip-v1",
        fields=base,
        required=[name for name, spec in base.items() if spec.required],
    )


def make_trace_points_df() -> pd.DataFrame:
    # u1 tiene 3 puntos -> 2 candidatos. El segundo par comparte location_ref y sirve para same_place.
    # u2 tiene 1 punto -> no produce candidato.
    return pd.DataFrame(
        {
            "point_id": ["p0", "p1", "p2", "p3"],
            "user_id": ["u1", "u1", "u1", "u2"],
            "time_utc": [
                "2026-01-01T08:00:00Z",
                "2026-01-01T08:10:00Z",
                "2026-01-01T09:00:00Z",
                "2026-01-01T08:05:00Z",
            ],
            "latitude": [-33.45, -33.46, -33.461, -33.40],
            "longitude": [-70.66, -70.67, -70.671, -70.65],
            "location_ref": ["A", "B", "B", "C"],
            "poi_cat": ["home", "work", "work", "shop"],
        }
    )


def make_trace_clusters_df() -> pd.DataFrame:
    # Dos bursts claros para un mismo usuario: (p0,p1) y (p2,p3)
    return pd.DataFrame(
        {
            "point_id": ["p0", "p1", "p2", "p3"],
            "user_id": ["u1", "u1", "u1", "u1"],
            "time_utc": [
                "2026-01-01T08:00:00Z",
                "2026-01-01T08:02:00Z",
                "2026-01-01T08:30:00Z",
                "2026-01-01T08:31:00Z",
            ],
            "latitude": [-33.45, -33.4501, -33.46, -33.4601],
            "longitude": [-70.66, -70.6601, -70.67, -70.6701],
            "location_ref": ["A", "A", "B", "B"],
            "poi_cat": ["home", "home", "work", "work"],
        }
    )


def make_trace_dataset(
    df: pd.DataFrame | None = None,
    *,
    validated: bool = True,
    dataset_id: str = "trace_ds_001",
    events: list[dict] | None = None,
) -> TraceDataset:
    if df is None:
        df = make_trace_points_df()
    if events is None:
        events = [{"op": "import_traces"}, {"op": "validate_traces"}]
    return TraceDataset(
        data=df.copy(),
        schema=make_trace_schema_min(),
        provenance={"source_name": "synthetic"},
        metadata={
            "dataset_id": dataset_id,
            "is_validated": validated,
            "events": deepcopy(events),
        },
    )


def make_points_options(**overrides) -> InferTripsOptions:
    base = dict(
        infer_mode="consecutive_points",
        strict=False,
        strict_domains=False,
        require_validated_traces=True,
        drop_invalid=True,
        h3_resolution=8,
        max_time_delta_s=None,
        min_time_delta_s=None,
        min_distance_m=None,
        cluster_radius_m=None,
        cluster_max_time_gap_s=None,
        propagate_trace_fields=None,
    )
    base.update(overrides)
    return InferTripsOptions(**base)


def make_cluster_options(**overrides) -> InferTripsOptions:
    base = dict(
        infer_mode="consecutive_clusters",
        strict=False,
        strict_domains=False,
        require_validated_traces=True,
        drop_invalid=True,
        h3_resolution=8,
        max_time_delta_s=None,
        min_time_delta_s=None,
        min_distance_m=None,
        cluster_radius_m=50.0,
        cluster_max_time_gap_s=300.0,
        propagate_trace_fields=None,
    )
    base.update(overrides)
    return InferTripsOptions(**base)

### Configuración visual

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## Bloque 1. Utilidades puras y helpers internos de uso general

### Test 1.1 - _normalize_optional_number

Se prueba un caso feliz y un caso fatal. Aquí interesa validar que los thresholds se normalicen bien y que un valor inválido efectivamente corte el flujo.

In [5]:
issues = []
value = _normalize_optional_number(
    issues,
    code="INF.OPTIONS.INVALID_MIN_DISTANCE",
    option_name="min_distance_m",
    value="150",
    allow_zero=True,
)
assert value == 150.0
assert issues == []
show_ok("1.1.a - normalización numérica válida")

issues = []
try:
    _normalize_optional_number(
        issues,
        code="INF.OPTIONS.INVALID_MIN_DISTANCE",
        option_name="min_distance_m",
        value=-1,
        allow_zero=True,
    )
    raise AssertionError("Se esperaba InferenceError para threshold inválido")
except InferenceError:
    assert_has_code(issues, "INF.OPTIONS.INVALID_MIN_DISTANCE")
show_ok("1.1.b - threshold inválido escala a InferenceError")

OK - 1.1.a - normalización numérica válida
OK - 1.1.b - threshold inválido escala a InferenceError


### Test 1.2 - _normalize_propagate_trace_fields

Se prueba un caso válido y un caso de colisión con campo reservado del output. Este segundo test es importante porque la propagación puede romper el contrato mínimo del TripDataset si no se controla.

In [6]:
traces = make_trace_dataset()

issues = []
prop_map = _normalize_propagate_trace_fields(
    issues,
    traces=traces,
    propagate_trace_fields={"poi_cat": "both", "location_ref": "origin"},
)
assert prop_map == {"poi_cat": "both", "location_ref": "origin"}
assert issues == []
show_ok("1.2.a - normalización de propagate_trace_fields válida")

issues = []
try:
    _normalize_propagate_trace_fields(
        issues,
        traces=traces,
        propagate_trace_fields={"time_utc": "origin"},  # -> origin_time_utc colisiona con campo reservado
    )
    raise AssertionError("Se esperaba InferenceError por colisión con campo reservado")
except InferenceError:
    assert_has_code(issues, "INF.PROPAGATION.RESERVED_TARGET_CONFLICT")
show_ok("1.2.b - colisión de propagación detectada")

OK - 1.2.a - normalización de propagate_trace_fields válida
OK - 1.2.b - colisión de propagación detectada


### Test 1.3 - _prepare_trace_workframe

Se verifica que:
1. no muta el dataframe original,
2. agrega _row_idx,
3. parsea time_utc,
4. coerciona latitude/longitude a numérico.

In [7]:
raw = make_trace_points_df()
raw_before = raw.copy(deep=True)

work = _prepare_trace_workframe(raw)

assert "_row_idx" in work.columns
assert pd.api.types.is_datetime64tz_dtype(work["time_utc"])
assert pd.api.types.is_numeric_dtype(work["latitude"])
assert pd.api.types.is_numeric_dtype(work["longitude"])
pd.testing.assert_frame_equal(raw, raw_before)
show_ok("1.3 - workframe tipado y sin mutación del input")

OK - 1.3 - workframe tipado y sin mutación del input


C:\Users\sebam\AppData\Local\Temp\ipykernel_15400\3913912361.py:7: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  assert pd.api.types.is_datetime64tz_dtype(work["time_utc"])


### Test 1.4 - _same_place_mask

Se verifica la regla same_place basada en location_ref, que luego afecta descartes en _evaluate_candidates.

In [8]:
candidates_df = pd.DataFrame(
    {
        "origin_location_ref": ["A", "B", None, "X"],
        "destination_location_ref": ["A", "C", "Z", "X"],
    }
)

mask = _same_place_mask(candidates_df)

assert mask.tolist() == [True, False, False, True]
show_ok("1.4 - same_place_mask detecta solo coincidencias reales")

OK - 1.4 - same_place_mask detecta solo coincidencias reales


### Test 1.5 - _safe_distance_meters y _safe_time_gap_seconds

Se prueban versiones puntuales seguras de distancia y gap temporal. Son helpers pequeños, pero muy centrales para el modo clusters y la evaluación de candidatos.

In [9]:
dist = _safe_distance_meters(-33.45, -70.66, -33.46, -70.67)
assert dist is not None and dist > 0

dist_none = _safe_distance_meters(-33.45, -70.66, None, -70.67)
assert dist_none is None

t1 = pd.Timestamp("2026-01-01T08:00:00Z")
t2 = pd.Timestamp("2026-01-01T08:10:00Z")
gap = _safe_time_gap_seconds(t1, t2)
assert gap == 600.0

gap_none = _safe_time_gap_seconds(pd.NaT, t2)
assert gap_none is None
show_ok("1.5 - helpers seguros de distancia y tiempo")

OK - 1.5 - helpers seguros de distancia y tiempo


### Test 1.6 - _expected_propagated_columns

Se valida la expansión origin_/destination_ a partir del mapping efectivo de propagación.

In [10]:
cols = _expected_propagated_columns({"poi_cat": "both", "location_ref": "origin"})
assert cols == ["origin_poi_cat", "destination_poi_cat", "origin_location_ref"]
show_ok("1.6 - expansión de columnas propagadas")

OK - 1.6 - expansión de columnas propagadas


### Test 1.7 - _normalize_output_categorical_field (bootstrap de dominio)

Se prueba un campo categórico declarado con DomainSpec.values=[] para verificar el caso bootstrap correcto del dominio efectivo.

In [11]:
series = pd.Series(["home", "home", pd.NA], dtype="string")
field_spec = FieldSpec(
    name="origin_poi_cat",
    dtype="categorical",
    required=False,
    domain=DomainSpec(values=[], extendable=True),
)

out_s, domain_eff, applied_map, issues, dtype_eff = _normalize_output_categorical_field(
    series,
    field_name="origin_poi_cat",
    field_spec=field_spec,
    value_mapping={"home": "residential"},
    strict_domains=False,
)

assert str(out_s.dtype) == "category"
assert out_s.astype("string").tolist() == ["residential", "residential", pd.NA]
assert domain_eff["values"] == ["residential"]
assert applied_map == {"home": "residential"}
assert dtype_eff == "categorical"
assert_has_code(issues, "MAP.VALUES.APPLIED")
assert_has_code(issues, "DOM.INFERENCE.APPLIED")
show_ok("1.7 - bootstrap categórico con value_correspondence")

OK - 1.7 - bootstrap categórico con value_correspondence


### Test 1.8 - _normalize_output_categorical_field (degradación por cardinalidad)

Se tensiona el caso donde el campo fue declarado categórico, pero la cardinalidad observada hace inviable tratarlo como dominio categórico de salida.

In [12]:
series = pd.Series([f"v{i}" for i in range(20)], dtype="string")
field_spec = FieldSpec(
    name="destination_poi_cat",
    dtype="categorical",
    required=False,
    domain=DomainSpec(values=[], extendable=True),
)

out_s, domain_eff, applied_map, issues, dtype_eff = _normalize_output_categorical_field(
    series,
    field_name="destination_poi_cat",
    field_spec=field_spec,
    value_mapping={},
    strict_domains=False,
)

assert str(out_s.dtype) == "string"
assert dtype_eff == "string"
assert domain_eff.get("degraded") is True
assert_has_code(issues, "DOM.INFERENCE.DEGRADED_TO_STRING")
show_ok("1.8 - degradación categórica por cardinalidad alta")

OK - 1.8 - degradación categórica por cardinalidad alta


### Test 1.9 - helpers de resumen y serialización

Se prueban _build_issues_summary, _summarize_prior_events, _time_range_summary y _to_json_safe.

In [13]:
issues = [
    make_issue("info", "INF.OK.SUMMARY"),
    make_issue("warning", "DOM.INFERENCE.DEGRADED_TO_STRING"),
    make_issue("warning", "DOM.INFERENCE.DEGRADED_TO_STRING"),
]

issues_summary = _build_issues_summary(issues)
assert issues_summary["counts"] == {"info": 1, "warning": 2, "error": 0}
assert issues_summary["counts_by_code"]["DOM.INFERENCE.DEGRADED_TO_STRING"] == 2

prior_events = [
    {"op": "import_traces"},
    {"op": "validate_traces"},
]
prior_summary = _summarize_prior_events(prior_events)
assert prior_summary["n_events"] == 2
assert prior_summary["last_event_op"] == "validate_traces"

time_range = _time_range_summary(pd.to_datetime(pd.Series(
    ["2026-01-01T00:00:00Z", "2026-01-02T00:00:00Z"], dtype="string"
), utc=True))
assert time_range == {
    "start": "2026-01-01T00:00:00Z",
    "end": "2026-01-02T00:00:00Z",
}

payload = {
    "ts": pd.Timestamp("2026-01-01T00:00:00Z"),
    "na": pd.NA,
    "nested": [1, pd.Timestamp("2026-01-02T00:00:00Z")],
}
safe_payload = _to_json_safe(payload)
json.dumps(safe_payload)  # no debe explotar
assert safe_payload["ts"] == "2026-01-01T00:00:00Z"
assert safe_payload["na"] is None
show_ok("1.9 - resumen de issues, prior events, time range y JSON-safe")

OK - 1.9 - resumen de issues, prior events, time range y JSON-safe


## Bloque 2. Tests de helpers principales

### Fixtures mínimas reutilizables para esta sección

In [14]:
trace_points_df = make_trace_points_df()
trace_clusters_df = make_trace_clusters_df()

traces_points = make_trace_dataset(trace_points_df, validated=True, dataset_id="trace_points_ds")
traces_unvalidated = make_trace_dataset(trace_points_df, validated=False, dataset_id="trace_points_ds_unvalidated")
traces_clusters = make_trace_dataset(trace_clusters_df, validated=True, dataset_id="trace_clusters_ds")

trip_schema_min = make_trip_schema_min(include_propagated_categoricals=False)
trip_schema_with_categoricals = make_trip_schema_min(include_propagated_categoricals=True)

### Grupo 2.1 - _resolve_infer_request

#### 2.1.1 - happy path mínimo en modo consecutive_points

In [15]:
issues = []
options_eff, parameters_eff, request_ctx = _resolve_infer_request(
    issues,
    traces=traces_points,
    trip_schema=trip_schema_with_categoricals,
    options=make_points_options(propagate_trace_fields={"poi_cat": "both"}),
    value_correspondence={"origin_poi_cat": {"home": "residential"}},
    provenance={"notebook": "helper_tests"},
)

assert issues == []
assert options_eff.infer_mode == "consecutive_points"
assert options_eff.propagate_trace_fields == {"poi_cat": "both"}
assert parameters_eff["value_correspondence_used"] is True
assert parameters_eff["validation_bypass_used"] is False
assert request_ctx["n_points_in"] == 4
assert request_ctx["n_users_in"] == 2
assert request_ctx["schema_version"] == "trip-v1"
show_ok("2.1.1 - request efectivo resuelto en modo points")

OK - 2.1.1 - request efectivo resuelto en modo points


#### 2.1.2 - bypass de validación explícito

Aquí se valida que, cuando `require_validated_traces=False`, la operación no aborte pero deje evidencia explícita del bypass.

In [16]:
issues = []
options_eff, parameters_eff, request_ctx = _resolve_infer_request(
    issues,
    traces=traces_unvalidated,
    trip_schema=trip_schema_min,
    options=make_points_options(require_validated_traces=False),
    value_correspondence=None,
    provenance=None,
)

assert_has_code(issues, "INF.PRECONDITION.VALIDATION_BYPASS_USED")
assert parameters_eff["validation_bypass_used"] is True
assert options_eff.require_validated_traces is False
show_ok("2.1.2 - bypass de validación queda trazado")

OK - 2.1.2 - bypass de validación queda trazado


#### 2.1.3 - contrato: en modo clusters deben exigirse cluster_radius_m y cluster_max_time_gap_s

Este test es especialmente valioso porque verifica una precondición contractual fuerte. Si falla, probablemente revela una brecha real de implementación.

In [17]:
issues = []
try:
    _resolve_infer_request(
        issues,
        traces=traces_points,
        trip_schema=trip_schema_min,
        options=make_cluster_options(cluster_radius_m=None, cluster_max_time_gap_s=None),
        value_correspondence=None,
        provenance=None,
    )
    raise AssertionError(
        "Se esperaba InferenceError: en modo consecutive_clusters el contrato exige "
        "cluster_radius_m y cluster_max_time_gap_s positivos."
    )
except InferenceError:
    # Si el helper quedó correcto, aquí debería entrar.
    # Si hoy no entra, este test te va a delatar una brecha respecto del cierre concreto.
    pass

show_ok("2.1.3 - test contractual de parámetros obligatorios para clusters")

OK - 2.1.3 - test contractual de parámetros obligatorios para clusters


### Grupo 2.2 - _build_point_candidates

#### 2.2.1 - forma pares consecutivos por usuario, respeta orden y arrastra campos frontera

In [18]:
issues = []
options_eff = make_points_options(propagate_trace_fields={"poi_cat": "both"})
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {"poi_cat": "both"},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    issues,
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.CANDIDATES.POINTS_MODE_APPLIED")
assert len(candidates) == 2  # u1 aporta 2 pares, u2 no aporta ninguno
assert candidates["origin_point_id"].tolist() == ["p0", "p1"]
assert candidates["destination_point_id"].tolist() == ["p1", "p2"]
assert "origin_poi_cat" in candidates.columns
assert "destination_poi_cat" in candidates.columns
assert candidates["same_place"].tolist() == [False, True]
show_ok("2.2.1 - pairing consecutivo y propagación frontera en modo points")

OK - 2.2.1 - pairing consecutivo y propagación frontera en modo points


### Grupo 2.3 - _build_sequential_clusters

#### 2.3.1 - colapso secuencial en dos clusters simples

In [19]:
options_eff = make_cluster_options(cluster_radius_m=50.0, cluster_max_time_gap_s=300.0)
clusters = _build_sequential_clusters(
    traces_clusters.data,
    options_eff=options_eff,
)

assert len(clusters) == 2
assert clusters["first_point_id"].tolist() == ["p0", "p2"]
assert clusters["last_point_id"].tolist() == ["p1", "p3"]
assert clusters["n_points"].tolist() == [2, 2]
show_ok("2.3.1 - construcción de clusters secuenciales")

OK - 2.3.1 - construcción de clusters secuenciales


### Grupo 2.4 - _build_cluster_candidates

#### 2.4.1 - usa puntos frontera correctos entre clusters consecutivos

In [20]:
issues = []
options_eff = make_cluster_options(
    cluster_radius_m=50.0,
    cluster_max_time_gap_s=300.0,
    propagate_trace_fields={"poi_cat": "both"},
)
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_clusters_df.columns),
    "propagate_trace_fields": {"poi_cat": "both"},
    "n_points_in": len(trace_clusters_df),
    "n_users_in": trace_clusters_df["user_id"].nunique(),
}

cluster_candidates = _build_cluster_candidates(
    issues,
    traces_clusters.data,
    clusters,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.CLUSTERS.MODE_APPLIED")
assert len(cluster_candidates) == 1

row = cluster_candidates.iloc[0]
assert row["origin_point_id"] == "p1"       # último del cluster origen
assert row["destination_point_id"] == "p2"  # primero del cluster destino
assert row["origin_poi_cat"] == "home"
assert row["destination_poi_cat"] == "work"
show_ok("2.4.1 - candidatos entre clusters usan puntos frontera correctos")

OK - 2.4.1 - candidatos entre clusters usan puntos frontera correctos


### Grupo 2.5 - _evaluate_candidates

#### 2.5.1 - descarta por max_time_delta_s y same_place, dejando summary consistente

In [21]:
issues = []
options_eff = make_points_options(
    max_time_delta_s=1_200,  # 20 min
    drop_invalid=True,
)
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

candidates_out, eval_info = _evaluate_candidates(
    issues,
    candidates,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.CANDIDATES.DROPPED_MAX_TIME_DELTA")
assert_has_code(issues, "INF.CANDIDATES.DROPPED_SAME_PLACE")
assert eval_info["n_candidates_in"] == 2
assert eval_info["n_candidates_dropped"] == 1
assert eval_info["n_trips_out"] == 1
assert eval_info["dropped_by_reason"]["max_time_delta_s"] == 1
assert eval_info["dropped_by_reason"]["same_place"] == 1
assert len(candidates_out) == 1
show_ok("2.5.1 - evaluación de thresholds y same_place")

OK - 2.5.1 - evaluación de thresholds y same_place


#### 2.5.2 - invalid candidate con drop_invalid=False deja evidencia y no pasa al output

Este test sirve para dejar explícita la semántica observada del helper actual.

In [22]:
issues = []
options_eff = make_points_options(drop_invalid=False)
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

candidates_bad = candidates.copy()

# Fuerzo un candidato inválido
candidates_bad.loc[0, "destination_point_id"] = pd.NA

# Neutralizo la regla same_place en el otro candidato
candidates_bad.loc[1, "destination_location_ref"] = "C"
candidates_bad["same_place"] = _same_place_mask(candidates_bad)

candidates_out, eval_info = _evaluate_candidates(
    issues,
    candidates_bad,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.CANDIDATES.INVALID_RETAINED")
assert eval_info["n_candidates_in"] == 2
assert len(candidates_out) == 1
show_ok("2.5.2 - evidencia por invalid candidate con drop_invalid=False")

OK - 2.5.2 - evidencia por invalid candidate con drop_invalid=False


### Grupo 2.6 - _materialize_trip_dataframe

#### 2.6.1 - construye núcleo mínimo del TripDataset y columnas propagadas

In [23]:
issues = []
options_eff = make_points_options(propagate_trace_fields={"poi_cat": "both"})
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {"poi_cat": "both"},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
candidates_out, _ = _evaluate_candidates(
    [],
    candidates,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

trip_df, materialization_info = _materialize_trip_dataframe(
    issues,
    candidates_out,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.PROPAGATION.APPLIED")
assert set(
    [
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
        "origin_time_utc",
        "destination_time_utc",
        "trip_id",
        "movement_seq",
        "origin_poi_cat",
        "destination_poi_cat",
    ]
).issubset(trip_df.columns)

assert trip_df["movement_id"].tolist() == ["m0"]
assert trip_df["trip_id"].tolist() == ["m0"]
assert trip_df["movement_seq"].tolist() == [0]
assert materialization_info["created_columns"] == ["origin_poi_cat", "destination_poi_cat"]
show_ok("2.6.1 - materialización canónica mínima")

OK - 2.6.1 - materialización canónica mínima


#### 2.6.2 - warning por soft width cap

Este test tensiona el guardrail de ancho del output sin necesitar cientos de columnas reales.

In [24]:
import pylondrina.transforms.inference as inference_mod

issues = []
options_eff = make_points_options(propagate_trace_fields={"poi_cat": "both"})
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {"poi_cat": "both"},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
candidates_out, _ = _evaluate_candidates(
    [],
    candidates,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

old_soft = inference_mod.TRIPDATASET_COLUMNS_SOFT_CAP
try:
    inference_mod.TRIPDATASET_COLUMNS_SOFT_CAP = 5
    _materialize_trip_dataframe(
        issues,
        candidates_out,
        options_eff=options_eff,
        request_ctx=request_ctx,
    )
    assert_has_code(issues, "INF.OUTPUT.SOFT_WIDTH_EXCEEDED")
finally:
    inference_mod.TRIPDATASET_COLUMNS_SOFT_CAP = old_soft

show_ok("2.6.2 - guardrail soft de ancho del output")

OK - 2.6.2 - guardrail soft de ancho del output


### Grupo 2.7 - _enrich_trip_dataframe

#### 2.7.1 - deriva H3 y normaliza categóricos propagados

In [25]:
issues = []
options_eff = make_points_options(
    h3_resolution=8,
    propagate_trace_fields={"poi_cat": "both"},
)
request_ctx = {
    "schema_version": "trip-v1",
    "available_fields": list(trace_points_df.columns),
    "propagate_trace_fields": {"poi_cat": "both"},
    "n_points_in": len(trace_points_df),
    "n_users_in": trace_points_df["user_id"].nunique(),
}

candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
candidates_out, _ = _evaluate_candidates(
    [],
    candidates,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
trip_df, _ = _materialize_trip_dataframe(
    [],
    candidates_out,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

trip_df_enriched, enrich_info = _enrich_trip_dataframe(
    issues,
    trip_df,
    trip_schema=trip_schema_with_categoricals,
    value_correspondence={
        "origin_poi_cat": {"home": "residential"},
        "destination_poi_cat": {"work": "employment"},
    },
    options_eff=options_eff,
    request_ctx=request_ctx,
)

assert_has_code(issues, "INF.H3.DERIVED")
assert_has_code(issues, "MAP.VALUES.APPLIED")
assert "origin_h3_index" in trip_df_enriched.columns
assert "destination_h3_index" in trip_df_enriched.columns
assert trip_df_enriched["origin_poi_cat"].astype("string").tolist() == ["residential"]
assert trip_df_enriched["destination_poi_cat"].astype("string").tolist() == ["employment"]
assert enrich_info["h3_meta"]["resolution"] == 8
assert enrich_info["dtype_effective"]["origin_poi_cat"] == "categorical"
show_ok("2.7.1 - enriquecimiento H3 + categóricos")

OK - 2.7.1 - enriquecimiento H3 + categóricos


#### 2.7.2 - degradación a string por cardinalidad alta en campo propagado

Este caso es útil para revisar la heurística de bootstrap categórico del output.

In [26]:
issues = []
trip_df_high_card = pd.DataFrame(
    {
        "movement_id": [f"m{i}" for i in range(20)],
        "user_id": ["u1"] * 20,
        "origin_longitude": [-70.66] * 20,
        "origin_latitude": [-33.45] * 20,
        "destination_longitude": [-70.67] * 20,
        "destination_latitude": [-33.46] * 20,
        "origin_time_utc": pd.to_datetime(["2026-01-01T08:00:00Z"] * 20, utc=True),
        "destination_time_utc": pd.to_datetime(["2026-01-01T08:10:00Z"] * 20, utc=True),
        "trip_id": [f"m{i}" for i in range(20)],
        "movement_seq": [0] * 20,
        "origin_poi_cat": [f"cat_{i}" for i in range(20)],
        "destination_poi_cat": ["stable"] * 20,
    }
)

trip_df_enriched, enrich_info = _enrich_trip_dataframe(
    issues,
    trip_df_high_card,
    trip_schema=trip_schema_with_categoricals,
    value_correspondence=None,
    options_eff=make_points_options(h3_resolution=8),
    request_ctx={
        "schema_version": "trip-v1",
        "available_fields": list(trip_df_high_card.columns),
        "propagate_trace_fields": {},
        "n_points_in": 0,
        "n_users_in": 0,
    },
)

display(issues)
assert_has_code(issues, "DOM.INFERENCE.DEGRADED_TO_STRING")
assert str(trip_df_enriched["origin_poi_cat"].dtype) == "string"
assert enrich_info["dtype_effective"]["origin_poi_cat"] == "string"
show_ok("2.7.2 - degradación a string por cardinalidad alta")

[Issue(level='info', code='INF.H3.DERIVED', message='Se derivaron origin_h3_index y destination_h3_index con h3_resolution=8.', field=None, source_field=None, row_count=None, details={'infer_mode': 'consecutive_points', 'strict': False, 'strict_domains': False, 'drop_invalid': True, 'require_validated_traces': True, 'h3_resolution': 8, 'max_time_delta_s': None, 'min_time_delta_s': None, 'min_distance_m': None, 'cluster_radius_m': None, 'cluster_max_time_gap_s': None, 'n_trips_out': 20, 'action': 'derived_h3'}),
 Issue(level='warning', code='DOM.INFERENCE.DEGRADED_TO_STRING', message="El campo 'origin_poi_cat' fue declarado categórico con DomainSpec.values vacío, pero su cardinalidad observada (20 únicos sobre 20 filas no nulas) supera el límite 1; se degradará a texto.", field='origin_poi_cat', source_field=None, row_count=None, details={'field': 'origin_poi_cat', 'n_rows_non_null': 20, 'n_unique_observed': 20, 'alpha': 0.05, 'k_max': 10000, 'cardinality_limit': 1, 'observed_values_sam

OK - 2.7.2 - degradación a string por cardinalidad alta


### Grupo 2.8 - _build_inference_outputs

#### 2.8.1 - cierra TripDataset + InferenceReport + metadata/evento/provenance

Aquí se valida el cierre completo del contrato observable del output.

In [27]:
issues = []

# 1) request efectivo
options_eff, parameters_eff, request_ctx = _resolve_infer_request(
    [],
    traces=traces_points,
    trip_schema=trip_schema_with_categoricals,
    options=make_points_options(propagate_trace_fields={"poi_cat": "both"}),
    value_correspondence={
        "origin_poi_cat": {"home": "residential"},
        "destination_poi_cat": {"work": "employment"},
    },
    provenance={"notebook": "helper_tests"},
)

# 2) candidatos
candidates = _build_point_candidates(
    [],
    traces_points.data,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
candidates_out, eval_info = _evaluate_candidates(
    [],
    candidates,
    options_eff=options_eff,
    request_ctx=request_ctx,
)

# 3) materialización y enriquecimiento
trip_df, materialization_info = _materialize_trip_dataframe(
    [],
    candidates_out,
    options_eff=options_eff,
    request_ctx=request_ctx,
)
trip_df_enriched, enrich_info = _enrich_trip_dataframe(
    [],
    trip_df,
    trip_schema=trip_schema_with_categoricals,
    value_correspondence={
        "origin_poi_cat": {"home": "residential"},
        "destination_poi_cat": {"work": "employment"},
    },
    options_eff=options_eff,
    request_ctx=request_ctx,
)

# 4) cierre final
trip_dataset, report = _build_inference_outputs(
    issues,
    traces=traces_points,
    trip_df=trip_df_enriched,
    trip_schema=trip_schema_with_categoricals,
    options_eff=options_eff,
    parameters_effective=parameters_eff,
    request_ctx=request_ctx,
    eval_info=eval_info,
    materialization_info=materialization_info,
    enrich_info=enrich_info,
    value_correspondence={
        "origin_poi_cat": {"home": "residential"},
        "destination_poi_cat": {"work": "employment"},
    },
    provenance={"notebook": "helper_tests"},
    clusters_df=None,
)

assert isinstance(trip_dataset, TripDataset)
assert isinstance(report, InferenceReport)
assert report.summary["infer_mode"] == "consecutive_points"
assert report.summary["n_points_in"] == 4
assert report.summary["n_trips_out"] == len(trip_dataset.data)

assert trip_dataset.field_correspondence == {}
assert trip_dataset.metadata["is_validated"] is False
assert trip_dataset.metadata["events"][0]["op"] == "infer_trips"
assert trip_dataset.metadata["events"][0]["parameters"]["infer_mode"] == "consecutive_points"
assert trip_dataset.metadata["temporal"]["tier"] == "tier_1"
assert trip_dataset.metadata["h3"]["resolution"] == 8

assert "derived_from" in trip_dataset.provenance
assert trip_dataset.provenance["derived_from"][0]["source_type"] == "traces"
assert trip_dataset.provenance["derived_from"][0]["dataset_id"] == "trace_points_ds"

# El output no debe copiar el historial completo de eventos del TraceDataset.
assert len(trip_dataset.metadata["events"]) == 1
assert trip_dataset.metadata["events"][0]["op"] == "infer_trips"

assert_has_code(issues, "INF.OK.SUMMARY")
show_ok("2.8.1 - cierre final de dataset, report, metadata, evento y provenance")

OK - 2.8.1 - cierre final de dataset, report, metadata, evento y provenance
